In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

STORE_ID = "496509ff-eaf8-4be8-bdb0-9fad838779ec"
TIENDA = "Vivanda"
current_date = datetime.now().strftime('%Y-%m-%d')

BASE_PRODUCTS = f"https://api.cord.pe/cord-rest/catalog/v2/stores/{STORE_ID}/products"
BASE_CATEGORIES = f"https://api.cord.pe/cord-rest/catalog/v2/stores/{STORE_ID}/categories"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

# =========================
# 1️⃣ OBTENER TODAS LAS CATEGORÍAS
# =========================

params = {"page": 0, "size": 2000}
r = requests.get(BASE_CATEGORIES, headers=HEADERS, params=params)
cats = r.json().get("items", [])

# =========================
# 2️⃣ FILTRAR SOLO SUBCATEGORÍAS DE INTERÉS (nivel 2)
# =========================

ALIMENTOS_ROOT = [
    "lacteos-y-huevos",
    "abarrotes", "quesos-y-fiambres","bebe-e-infantil","carnes-aves-y-pescados","congelados","frutas-y-verduras",
    "pollo-rostizado-y-comidas-preparadas","panaderia-y-pasteleria","bebidas","desayunos"
]

food_categories = []

for c in cats:
    permalink = c.get("seo", {}).get("permalink", "")
    if not permalink:
        continue
    partes = permalink.split("/")
    # Solo subcategorías hijas (nivel 2)
    if len(partes) == 2 and partes[0] in ALIMENTOS_ROOT:
        food_categories.append({
            "id": c.get("id"),
            "categoria": partes[0].replace("-", " ").title(),
            "subcategoria": partes[1].replace("-", " ").title()
        })

print("Total subcategorías filtradas:", len(food_categories))

# =========================
# 3️⃣ SCRAPEAR PRODUCTOS
# =========================

all_products = []

for cat in food_categories:
    category_id = cat["id"]
    page = 0

    print(f"Procesando categoría: {cat['categoria']}/{cat['subcategoria']}")

    while True:
        params = {
            "page": page,
            "size": 24,
            "includeFacets": "true",
            "categoryIds": category_id
        }

        r = requests.get(BASE_PRODUCTS, headers=HEADERS, params=params)
        data = r.json()
        items = data.get("items", [])

        if not items:
            break

        for item in items:
            # Marca
            marca = item.get("brand", {}).get("name")

            # Precios
            precio_lista = None
            precio_final = None
            descuento_monto = None
            descuento_porcentaje = None

            pricing = item.get("pricing", {})
            if pricing:
                precio_lista = pricing.get("regularPrice", {}).get("amount")
                precio_final = pricing.get("finalPrice", {}).get("amount")

                # Calcular descuento si corresponde
                if precio_lista and precio_final and precio_lista > precio_final:
                    descuento_monto = round(precio_lista - precio_final, 2)
                    descuento_porcentaje = round(((precio_lista - precio_final) / precio_lista) * 100, 2)

            # Presentación (si existe)
            presentacion = item.get("name")

            # Disponible
            disponible = item.get("stock", {}).get("available", False)
            disponible = "VERDADERO" if disponible else "FALSO"

            all_products.append({
                "CATEGORIA": cat["categoria"],
                "SUBCATEGORIA": cat["subcategoria"],
                "sku": item.get("sku"),
                "nombre": item.get("name"),
                "marca": marca,
                "precio_lista": precio_lista,
                "precio_final": precio_final,
                "descuento_monto": descuento_monto,
                "descuento_porcentaje": descuento_porcentaje,
                "presentacion": presentacion,
                "disponible": disponible,
                "fecha_scraping": current_date,
                "tienda": TIENDA
            })

        page += 1
        time.sleep(0.3)

print("\nTotal productos recolectados:", len(all_products))

# =========================
# 4️⃣ EXPORTAR A EXCEL
# =========================

df = pd.DataFrame(all_products)


cols = [
    "CATEGORIA","SUBCATEGORIA","sku","nombre","marca",
    "precio_lista","precio_final","descuento_monto","descuento_porcentaje",
    "presentacion","disponible","fecha_scraping","tienda"
]
df = df[cols]

# Eliminar duplicados por SKU
df.drop_duplicates(subset=["sku"], inplace=True)

df.to_excel("supermercado_alimentos.xlsx", index=False)

print("Excel generado correctamente ✅")

Total subcategorías filtradas: 65
Procesando categoría: Panaderia Y Pasteleria/Panetones
Procesando categoría: Pollo Rostizado Y Comidas Preparadas/Pizzas Y Pastas Frescas
Procesando categoría: Pollo Rostizado Y Comidas Preparadas/Pollo Rostizado
Procesando categoría: Abarrotes/Canastas Navidenas
Procesando categoría: Lacteos Y Huevos/Huevos
Procesando categoría: Pollo Rostizado Y Comidas Preparadas/Tamales Y Humitas
Procesando categoría: Bebidas/Gaseosas
Procesando categoría: Desayunos/Cafe E Infusiones
Procesando categoría: Frutas Y Verduras/Frutas
Procesando categoría: Bebidas/Aguas
Procesando categoría: Carnes Aves Y Pescados/Cerdo
Procesando categoría: Pollo Rostizado Y Comidas Preparadas/Cremas Salsas Y Condimentos A Granel
Procesando categoría: Quesos Y Fiambres/Quesos Procesados
Procesando categoría: Abarrotes/Galletas Y Golosinas
Procesando categoría: Carnes Aves Y Pescados/Pollo
Procesando categoría: Pollo Rostizado Y Comidas Preparadas/Comidas Preparadas
Procesando categoría